# Tarea 2.4 · Métricas de rendimiento y escalabilidad

**Curso:** Herramientas de Big Data · Saber 2: Procesamiento distribuido
**Proyecto:** Emergencias de la Línea 123 de Bogotá (julio, agosto y septiembre de 2026)
**Actividad:** Actividad 2 · Diseño de arquitectura y PoC (evaluativa)

**Objetivos**
- Ejecutar el PoC 3 veces y medir tiempo total (s), memoria pico (%), throughput (registros/min) y tasa de éxito de validaciones (%).
- Comparar con el mismo proceso en Pandas.
- Estimar la escalabilidad con volúmenes de 1 y 2 años (datos replicados) y proyectarla.

**Prerrequisitos**
- `03_T2.2_PoC_PySpark.ipynb` ejecutado (tabla de códigos y Parquet en Drive).

**Tiempo estimado de ejecución:** ~6 minutos (Colab gratuito, solo CPU)

**Dataset:** «Llamadas de urgencias y emergencias que ingresan a través de la línea 123», Secretaría Distrital de
Salud, portal Datos Abiertos Bogotá (CC BY 4.0). Tres CSV mensuales (27 434 registros, separador `;`).

## 1. Configuración del entorno

In [ ]:
# 1.1 Conexión con Google Drive (los datos y los resultados viven en MyDrive/BigData_U2)
import os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 1.2 Librerías
import time, json, inspect, threading, platform, gc, shutil
import numpy as np
import pandas as pd
import polars as pl
import psutil
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

os.environ['TZ'] = 'America/Bogota'   # hora local en los logs
time.tzset()

# 1.3 Rutas de trabajo (una sola carpeta del proyecto en Drive)
BASE = '/content/drive/MyDrive/BigData_U2'
DATOS = f'{BASE}/datos'
RESULTADOS = f'{BASE}/resultados'
MEDICIONES = f'{RESULTADOS}/mediciones'
for ruta in (RESULTADOS, MEDICIONES, f'{RESULTADOS}/logs', f'{RESULTADOS}/parquet'):
    os.makedirs(ruta, exist_ok=True)

ARCHIVOS = ['llamadas123_julio_2026.csv', 'llamadaslinea123agosto2026.csv', 'llamadaslinea123Septiembre.csv']
RUTAS = [f'{DATOS}/{a}' for a in ARCHIVOS]
COLUMNAS = ['NUMERO_INCIDENTE', 'FECHA_INICIO_DESPLAZAMIENTO_MOVIL', 'CODIGO_LOCALIDAD', 'LOCALIDAD', 'EDAD',
            'UNIDAD', 'GENERO', 'TIPO_INCIDENTE', 'PRIORIDAD_FINAL', 'RECEPCION']
faltan = [a for a in ARCHIVOS if not os.path.exists(f'{DATOS}/{a}')]
assert not faltan, f'Sube a {DATOS} los archivos: {faltan}'

print('✓ Drive montado y librerías importadas')
print(f'   Python {platform.python_version()} · pandas {pd.__version__} · polars {pl.__version__} · '
      f'PySpark {pyspark.__version__}')

In [ ]:
# 1.4 Funciones de monitoreo (notebook 02 del curso: get_memory_info / print_memory_status)
def get_memory_info():
    """Obtiene información sobre el uso de memoria del sistema."""
    memory = psutil.virtual_memory()
    return {'total_gb': round(memory.total / 1024**3, 2), 'disponible_gb': round(memory.available / 1024**3, 2),
            'usado_gb': round(memory.used / 1024**3, 2), 'porcentaje': memory.percent}


def print_memory_status(titulo='Estado de memoria'):
    mem = get_memory_info()
    print(f"\n📊 {titulo}: {mem['usado_gb']} de {mem['total_gb']} GB usados ({mem['porcentaje']:.1f} %), "
          f"{mem['disponible_gb']} GB disponibles")


class MonitorMemoria:
    """Muestrea psutil.virtual_memory().percent cada 50 ms para capturar el PICO de memoria durante una operación."""

    def __init__(self, intervalo=0.05):
        self.intervalo, self.pico, self._activo = intervalo, 0.0, False

    def _muestrear(self):
        while self._activo:
            self.pico = max(self.pico, psutil.virtual_memory().percent)
            time.sleep(self.intervalo)

    def __enter__(self):
        self.pico, self._activo = psutil.virtual_memory().percent, True
        self._hilo = threading.Thread(target=self._muestrear, daemon=True)
        self._hilo.start()
        return self

    def __exit__(self, *exc):
        self._activo = False
        self._hilo.join()


def measure_performance(func, *args, **kwargs):
    """Mide tiempo y memoria de una operación (adaptación de measure_operation, notebook 04 del curso).

    - Tiempo: time.time() antes y después.
    - Memoria: psutil.virtual_memory() (porcentaje del sistema antes y pico durante la operación) y el cambio de
      RSS del proceso Python. El % del sistema incluye la JVM de Spark, que corre en otro proceso.
    """
    proceso = psutil.Process()
    rss_antes = proceso.memory_info().rss / 1024**2
    mem_antes = psutil.virtual_memory().percent
    with MonitorMemoria() as monitor:
        inicio = time.time()
        resultado = func(*args, **kwargs)
        fin = time.time()
    return {'result': resultado, 'time_seconds': fin - inicio, 'mem_pct_antes': mem_antes,
            'mem_pct_pico': monitor.pico, 'memory_delta_mb': proceso.memory_info().rss / 1024**2 - rss_antes}


print('✓ Funciones de medición configuradas')
print_memory_status('Memoria inicial')

In [ ]:
# Helper del notebook 02 del curso (create_optimized_spark_session), con la zona horaria de Bogotá
def create_optimized_spark_session(app_name='Emergencias123', memory_fraction=0.7):
    """Crea una SparkSession optimizada para Google Colab."""
    num_cores = os.cpu_count()
    driver_memory = f"{max(1, int(get_memory_info()['disponible_gb'] * memory_fraction))}g"   # mínimo 1 GB
    return (SparkSession.builder
            .appName(app_name)
            .master(f'local[{num_cores}]')
            .config('spark.driver.memory', driver_memory)
            .config('spark.driver.maxResultSize', '2g')                           # configuración de la guía
            .config('spark.sql.adaptive.enabled', 'true')
            .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
            .config('spark.sql.adaptive.skewJoin.enabled', 'true')                # configuración de la guía
            .config('spark.sql.adaptive.advisoryPartitionSizeInBytes', '64MB')   # configuración de la guía
            .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
            .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
            .config('spark.sql.session.timeZone', 'America/Bogota')
            .config('spark.ui.showConsoleProgress', 'false')
            .getOrCreate())


m = measure_performance(create_optimized_spark_session)
spark = m['result']
TIEMPO_INICIO_SPARK = m['time_seconds']
print(f"✅ SparkSession creada en {TIEMPO_INICIO_SPARK:.2f} s")
print(f"   - Master: {spark.sparkContext.master}")
print(f"   - Paralelismo por defecto: {spark.sparkContext.defaultParallelism}")
print(f"   - Driver memory: {spark.conf.get('spark.driver.memory')}")
print(f"   - AQE: {spark.conf.get('spark.sql.adaptive.enabled')} · skewJoin: "
      f"{spark.conf.get('spark.sql.adaptive.skewJoin.enabled')} · Serializador: Kryo · Arrow: activado")
print('\n💡 En Colab, Spark corre en modo local: driver y ejecutores comparten una JVM en un solo nodo (no es un clúster).')

## 2. El pipeline del PoC y su equivalente en Pandas

Es el **mismo** `PipelineEmergencias` del notebook 03 (tarea 2.2). Para comparar en igualdad de condiciones,
`pipeline_pandas` hace exactamente lo mismo en Pandas: mismas reglas de limpieza, mismos 12 controles y las mismas
salidas en Parquet. Aquí ambas escriben en el disco local del VM, para que la latencia de Drive no afecte la comparación.

In [ ]:
# Normalización común a las tres herramientas: mayúsculas, sin tildes y con espacios simples.
# Necesaria porque los datos traen «USAQUÉN» con tilde y un tipo con doble espacio («AMESUI  - ...»).
CON_TILDE, SIN_TILDE = 'ÁÉÍÓÚÜÑ', 'AEIOUUN'
TABLA_TILDES = str.maketrans(CON_TILDE, SIN_TILDE)
LOCALIDADES = ['USAQUEN', 'CHAPINERO']

ESQUEMA = StructType([StructField(c, StringType(), True) for c in COLUMNAS])   # esquema explícito: nada se infiere
MESES = {'julio': '2026-07', 'agosto': '2026-08', 'septiembre': '2026-09'}
OBSERVACIONES = ['fecha_invalida', 'fecha_fuera_de_mes', 'localidad_sin_dato', 'edad_nula', 'genero_nulo', 'recepcion_nula',
                 'recepcion_antes_de_desplazamiento']
# Edad homologada a años según UNIDAD (valores reales en los datos: Años, Meses, Dias, Horas; «N/A» ya es nulo)
FACTOR_EDAD = {'ANOS': 1.0, 'MESES': 1 / 12, 'DIAS': 1 / 365, 'HORAS': 1 / 8760}


def periodo_de(nombre_archivo):
    """Periodo (mes) del archivo de origen, tomado de su nombre."""
    nombre = nombre_archivo.lower()
    return next(p for mes, p in MESES.items() if mes in nombre)


def evaluar_controles(s):
    """Controles de calidad comunes a PySpark y Pandas a partir de los conteos `s` del pipeline.

    Devuelve la tabla de controles y la tasa de éxito = aprobados / evaluados × 100.
    """
    controles = [
        ('V01', 'Se leyeron los 3 archivos', s['archivos'] == 3, s['archivos']),
        ('V02', 'Columnas requeridas presentes', s['columnas_ok'], s['columnas_ok']),
        ('V03', 'Leídos = aceptados + duplicados exactos apartados', s['leidos'] == s['aceptados'] + s['duplicados'],
         f"{s['leidos']} = {s['aceptados']} + {s['duplicados']}"),
        ('V04', 'Fechas de desplazamiento válidas (0 sin convertir)', s['fechas_invalidas'] == 0, s['fechas_invalidas']),
        ('V05', 'Fechas dentro del mes del archivo (+1 día de tolerancia)', s['fuera_ventana'] == 0, s['fuera_ventana']),
        ('V06', 'Todos los códigos de tipo están en el catálogo', s['sin_catalogo'] == 0, s['sin_catalogo']),
        ('V07', 'El join no cambia el número de filas', s['filas_antes_join'] == s['filas_despues_join'],
         f"{s['filas_antes_join']} → {s['filas_despues_join']}"),
        ('V08', 'Cada localidad tiene un único código', s['localidades_inconsistentes'] == 0, s['localidades_inconsistentes']),
        ('V09', 'Edades dentro de 0–120', s['edades_fuera_rango'] == 0, s['edades_fuera_rango']),
        ('V10', 'Sin duplicados exactos remanentes', s['duplicados_remanentes'] == 0, s['duplicados_remanentes']),
        ('V11', 'El Parquet releído tiene las filas aceptadas', s['parquet_filas'] == s['aceptados'], s['parquet_filas']),
        ('V12', 'RECEPCION nunca es anterior al inicio del desplazamiento', s['recepcion_antes'] == 0, s['recepcion_antes']),
    ]
    tabla = pd.DataFrame(controles, columns=['control', 'descripcion', 'aprobado', 'valor'])
    return tabla, round(tabla['aprobado'].mean() * 100, 1)


class PipelineEmergencias:
    """Pipeline robusto en PySpark (adaptación de RobustPipeline del notebook 04 del curso).

    extract → transform → validate → load, con log en archivo .txt y métricas de cada etapa.
    Regla de limpieza del proyecto: solo se apartan los duplicados EXACTOS (100 % de los atributos iguales) y se
    guardan aparte. Los nulos y demás anomalías se conservan y se marcan en `observacion_calidad`. Nunca se imputa.
    """

    def __init__(self, spark, nombre, destino, log_file):
        self.spark, self.nombre, self.destino, self.log_file = spark, nombre, destino, log_file
        self.metrics = {}
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        open(log_file, 'w', encoding='utf-8').close()

    def log(self, mensaje, nivel='INFO', mostrar=True):
        entrada = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [{nivel}] {mensaje}"
        if mostrar:
            print('   ' + entrada)
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(entrada + '\n')

    def extract(self, rutas):
        inicio = time.time()
        df = (self.spark.read.csv(rutas, schema=ESQUEMA, sep=';', header=True, encoding='UTF-8')
              .withColumn('archivo_origen', F.col('_metadata.file_name')))
        periodo = F.lit(None).cast('string')
        for mes, p in MESES.items():
            periodo = F.when(F.lower(F.col('archivo_origen')).contains(mes), p).otherwise(periodo)
        df = df.withColumn('periodo', periodo).cache()
        self.metrics['leidos'] = df.count()
        self.metrics['archivos'] = df.select('archivo_origen').distinct().count()
        self.metrics['extract_s'] = time.time() - inicio
        self.log(f"Extract: {self.metrics['leidos']:,} registros de {self.metrics['archivos']} archivos")
        return df

    def transform(self, df, catalogo):
        inicio = time.time()
        # 1) Duplicados exactos (todas las columnas originales iguales): se conserva la primera copia
        ventana = Window.partitionBy(*COLUMNAS).orderBy('archivo_origen')
        df = df.withColumn('_n', F.row_number().over(ventana))
        duplicados = df.filter('_n > 1').drop('_n').cache()
        df = df.filter('_n = 1').drop('_n')
        # 2) Nulos: «N/A» es un valor faltante; se convierte en null (sin imputar)
        for c in ('EDAD', 'UNIDAD', 'GENERO'):
            df = df.withColumn(c, F.when(F.trim(F.col(c)) == 'N/A', None).otherwise(F.col(c)))
        # 3) Tipos y 4) estandarización de textos y códigos
        norm = lambda c: F.trim(F.regexp_replace(F.translate(F.upper(F.col(c)), CON_TILDE, SIN_TILDE), r'\s+', ' '))
        fmt = F.lit('yyyy-MM-dd HH:mm:ss')
        df = (df
              .withColumn('fecha_inicio', F.try_to_timestamp(F.col('FECHA_INICIO_DESPLAZAMIENTO_MOVIL'), fmt))
              .withColumn('fecha_recepcion', F.try_to_timestamp(F.col('RECEPCION'), fmt))
              .withColumn('cod_localidad', F.expr('try_cast(CODIGO_LOCALIDAD AS INT)'))
              .withColumn('edad_num', F.expr('try_cast(EDAD AS INT)'))
              .withColumn('localidad_norm', norm('LOCALIDAD'))
              .withColumn('codigo_tipo', F.trim(F.split(norm('TIPO_INCIDENTE'), ' - ').getItem(0)))
              .withColumn('prioridad', norm('PRIORIDAD_FINAL'))
              .withColumn('genero_norm', norm('GENERO')))
        # 4b) Edad en años según la unidad (meses, días y horas se convierten; una unidad desconocida queda nula)
        factor = F.lit(None).cast('double')
        for unidad, valor in FACTOR_EDAD.items():
            factor = F.when(norm('UNIDAD') == unidad, F.lit(valor)).otherwise(factor)
        # 4c) Minutos entre el inicio del desplazamiento y RECEPCION. No es el «tiempo de respuesta»: en los datos
        #     RECEPCION siempre es posterior al despacho (mediana ~3 h), así que se reporta con su nombre literal.
        df = (df
              .withColumn('edad_anios', F.round(F.col('edad_num') * factor, 2))
              .withColumn('min_desplazamiento_a_recepcion',
                          F.round((F.col('fecha_recepcion').cast('long') - F.col('fecha_inicio').cast('long')) / 60, 1)))
        # 5) Observaciones de calidad: se marcan, no se borran
        inicio_mes = F.to_timestamp(F.concat(F.col('periodo'), F.lit('-01')))
        fin_ventana = F.date_add(F.add_months(inicio_mes, 1), 1)
        condiciones = {
            'fecha_invalida': F.col('fecha_inicio').isNull(),
            'fecha_fuera_de_mes': (F.col('fecha_inicio') < inicio_mes) | (F.col('fecha_inicio') >= fin_ventana),
            'localidad_sin_dato': F.col('cod_localidad').isNull(),
            'edad_nula': F.col('edad_num').isNull(),
            'genero_nulo': F.col('genero_norm').isNull(),
            'recepcion_nula': F.col('fecha_recepcion').isNull(),
            'recepcion_antes_de_desplazamiento': F.col('min_desplazamiento_a_recepcion') < 0}
        obs = F.concat_ws('; ', *[F.when(condiciones[n], F.lit(n)) for n in OBSERVACIONES])
        df = df.withColumn('observacion_calidad', F.when(obs == '', 'OK').otherwise(obs))
        # 6) Enriquecimiento: broadcast join con la tabla de códigos (23 filas)
        self.metrics['filas_antes_join'] = df.count()
        df = df.join(F.broadcast(catalogo), on='codigo_tipo', how='left').cache()
        self.metrics['filas_despues_join'] = df.count()
        self.metrics['transform_s'] = time.time() - inicio
        self.log(f"Transform: {self.metrics['filas_despues_join']:,} aceptados; duplicados exactos apartados: "
                 f"{duplicados.count():,}")
        return df, duplicados

    def validate(self, df, duplicados):
        inicio = time.time()
        s = {k: self.metrics[k] for k in ('archivos', 'leidos', 'filas_antes_join', 'filas_despues_join')}
        s['aceptados'], s['duplicados'] = df.count(), duplicados.count()
        s['columnas_ok'] = all(c in df.columns for c in COLUMNAS)
        fila = df.agg(
            F.sum(F.col('fecha_inicio').isNull().cast('int')).alias('fechas_invalidas'),
            F.sum(F.col('observacion_calidad').contains('fecha_fuera_de_mes').cast('int')).alias('fuera_ventana'),
            F.sum(F.col('descripcion_tipo').isNull().cast('int')).alias('sin_catalogo'),
            F.sum(((F.col('edad_anios') < 0) | (F.col('edad_anios') > 120)).cast('int')).alias('edades_fuera_rango'),
            F.sum((F.col('min_desplazamiento_a_recepcion') < 0).cast('int')).alias('recepcion_antes'),
        ).first().asDict()
        s.update({k: int(v or 0) for k, v in fila.items()})
        s['localidades_inconsistentes'] = (df.groupBy('localidad_norm').agg(F.countDistinct('cod_localidad').alias('n'))
                                           .filter('n > 1').count())
        s['duplicados_remanentes'] = s['aceptados'] - df.select(*COLUMNAS).distinct().count()
        self.stats = s
        self.metrics['validate_s'] = time.time() - inicio
        self.log(f"Validate: controles calculados sobre {s['aceptados']:,} registros")
        return s

    def load(self, df, duplicados):
        inicio = time.time()
        # Particionamiento mes/localidad que pide la guía: periodo (mes del archivo) y código de localidad.
        # repartition por las mismas columnas → un archivo por carpeta (sin archivos sueltos de más).
        (df.repartition('periodo', 'cod_localidad')
           .write.mode('overwrite').partitionBy('periodo', 'cod_localidad').parquet(f'{self.destino}/detalle'))
        duplicados.write.mode('overwrite').parquet(f'{self.destino}/duplicados_eliminados')
        (df.groupBy('periodo', 'localidad_norm', 'codigo_tipo', 'descripcion_tipo').count()
           .write.mode('overwrite').parquet(f'{self.destino}/agregados'))
        self.stats['parquet_filas'] = self.spark.read.parquet(f'{self.destino}/detalle').count()
        self.metrics['load_s'] = time.time() - inicio
        self.log(f"Load: Parquet particionado por periodo/cod_localidad en {self.destino}")

    def run(self, rutas, catalogo):
        self.log('=' * 60)
        self.log(f'INICIANDO PIPELINE: {self.nombre}')
        inicio = time.time()
        try:
            df = self.extract(rutas)
            limpio, duplicados = self.transform(df, catalogo)
            self.validate(limpio, duplicados)
            self.load(limpio, duplicados)
            errores = 0
        except Exception as e:
            self.log(f'Error en el pipeline: {e}', 'ERROR')
            raise
        self.metrics['tiempo_total_s'] = time.time() - inicio
        controles, tasa = evaluar_controles(self.stats)
        self.metrics.update(aceptados=self.stats['aceptados'], duplicados=self.stats['duplicados'],
                            tasa_validaciones_pct=tasa, errores=errores,
                            controles_fallidos=int((~controles['aprobado']).sum()))
        self.log('-' * 60)
        self.log('REPORTE DE VALIDACIÓN')
        for _, c in controles.iterrows():
            self.log(f"{c['control']} {'OK   ' if c['aprobado'] else 'FALLA'} {c['descripcion']} (valor: {c['valor']})",
                     'INFO' if c['aprobado'] else 'WARNING', mostrar=False)
        self.log(f"Registros procesados: {self.metrics['leidos']:,} leídos · {self.metrics['aceptados']:,} aceptados · "
                 f"{self.metrics['duplicados']:,} duplicados exactos apartados")
        self.log(f"Errores de ejecución: {errores} · Tasa de éxito de validaciones: {tasa} % "
                 f"({len(controles) - self.metrics['controles_fallidos']}/{len(controles)} controles)")
        self.log(f"Tiempo de ejecución: {self.metrics['tiempo_total_s']:.2f} s "
                 f"(extract {self.metrics['extract_s']:.2f} · transform {self.metrics['transform_s']:.2f} · "
                 f"validate {self.metrics['validate_s']:.2f} · load {self.metrics['load_s']:.2f})")
        self.spark.catalog.clearCache()
        return controles


def leer_catalogo_spark(spark):
    return (spark.read.csv(f'{RESULTADOS}/catalogo_tipos_derivado.csv', header=True)
            .select('codigo_tipo', 'descripcion_tipo'))


print('✓ PipelineEmergencias definido')

In [ ]:
def pipeline_pandas(rutas, destino, catalogo_pd):
    """El MISMO alcance del PoC en Pandas (mismas reglas, controles y salidas) para comparar en igualdad."""
    inicio = time.time()
    partes = []
    for r in rutas:
        d = pd.read_csv(r, sep=';', dtype=str, encoding='utf-8-sig', keep_default_na=False, na_values=[''])
        d['archivo_origen'] = os.path.basename(r)
        partes.append(d)
    df = pd.concat(partes, ignore_index=True)
    df['periodo'] = df['archivo_origen'].map(periodo_de)
    s = {'archivos': df['archivo_origen'].nunique(), 'leidos': len(df)}
    es_dup = df.duplicated(subset=COLUMNAS, keep='first')
    duplicados, df = df[es_dup], df[~es_dup].copy()
    for c in ('EDAD', 'UNIDAD', 'GENERO'):
        df[c] = df[c].where(df[c].str.strip() != 'N/A')
    norm = lambda serie: serie.str.upper().str.translate(TABLA_TILDES).str.replace(r'\s+', ' ', regex=True).str.strip()
    df['fecha_inicio'] = pd.to_datetime(df['FECHA_INICIO_DESPLAZAMIENTO_MOVIL'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df['fecha_recepcion'] = pd.to_datetime(df['RECEPCION'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df['cod_localidad'] = pd.to_numeric(df['CODIGO_LOCALIDAD'], errors='coerce').astype('Int64')
    df['edad_num'] = pd.to_numeric(df['EDAD'], errors='coerce').astype('Int64')
    df['localidad_norm'] = norm(df['LOCALIDAD'])
    df['codigo_tipo'] = norm(df['TIPO_INCIDENTE']).str.split(' - ', n=1).str[0].str.strip()
    df['prioridad'] = norm(df['PRIORIDAD_FINAL'])
    df['genero_norm'] = norm(df['GENERO'])
    df['edad_anios'] = (df['edad_num'].astype('float') * norm(df['UNIDAD']).map(FACTOR_EDAD)).round(2)
    df['min_desplazamiento_a_recepcion'] = ((df['fecha_recepcion'] - df['fecha_inicio']).dt.total_seconds() / 60).round(1)
    inicio_mes = pd.to_datetime(df['periodo'] + '-01')
    fin_ventana = inicio_mes + pd.DateOffset(months=1) + pd.Timedelta(days=1)
    condiciones = {
        'fecha_invalida': df['fecha_inicio'].isna(),
        'fecha_fuera_de_mes': (df['fecha_inicio'] < inicio_mes) | (df['fecha_inicio'] >= fin_ventana),
        'localidad_sin_dato': df['cod_localidad'].isna(),
        'edad_nula': df['edad_num'].isna(),
        'genero_nulo': df['genero_norm'].isna(),
        'recepcion_nula': df['fecha_recepcion'].isna(),
        'recepcion_antes_de_desplazamiento': df['min_desplazamiento_a_recepcion'] < 0}
    obs = pd.Series('', index=df.index)
    for n in OBSERVACIONES:
        obs = obs + np.where(condiciones[n], n + '; ', '')
    obs = obs.str.rstrip('; ')
    df['observacion_calidad'] = obs.where(obs != '', 'OK')
    s['filas_antes_join'] = len(df)
    df = df.merge(catalogo_pd, on='codigo_tipo', how='left', validate='m:1')
    s['filas_despues_join'] = len(df)
    s['aceptados'], s['duplicados'] = len(df), len(duplicados)
    s['columnas_ok'] = all(c in df.columns for c in COLUMNAS)
    s['fechas_invalidas'] = int(df['fecha_inicio'].isna().sum())
    s['fuera_ventana'] = int(df['observacion_calidad'].str.contains('fecha_fuera_de_mes').sum())
    s['sin_catalogo'] = int(df['descripcion_tipo'].isna().sum())
    s['edades_fuera_rango'] = int(((df['edad_anios'] < 0) | (df['edad_anios'] > 120)).fillna(False).sum())
    s['recepcion_antes'] = int((df['min_desplazamiento_a_recepcion'] < 0).sum())
    s['localidades_inconsistentes'] = int((df.groupby('localidad_norm')['cod_localidad'].nunique() > 1).sum())
    s['duplicados_remanentes'] = int(df.duplicated(subset=COLUMNAS).sum())
    shutil.rmtree(destino, ignore_errors=True)
    os.makedirs(destino, exist_ok=True)
    # La clave de partición va como texto: pyarrow no relee una partición Int64 con nulos (la fila SIN_D)
    (df.assign(cod_localidad=df['cod_localidad'].astype('string'))
       .to_parquet(f'{destino}/detalle', partition_cols=['periodo', 'cod_localidad'], index=False))
    duplicados.to_parquet(f'{destino}/duplicados_eliminados.parquet', index=False)
    (df.groupby(['periodo', 'localidad_norm', 'codigo_tipo', 'descripcion_tipo'], dropna=False).size()
       .reset_index(name='count').to_parquet(f'{destino}/agregados.parquet', index=False))
    s['parquet_filas'] = len(pd.read_parquet(f'{destino}/detalle', columns=['NUMERO_INCIDENTE']))
    controles, tasa = evaluar_controles(s)
    return {'leidos': s['leidos'], 'aceptados': s['aceptados'], 'duplicados': s['duplicados'],
            'tiempo_total_s': time.time() - inicio, 'tasa_validaciones_pct': tasa, 'errores': 0,
            'controles_fallidos': int((~controles['aprobado']).sum())}


catalogo_pd = pd.read_csv(f'{RESULTADOS}/catalogo_tipos_derivado.csv', dtype=str)[['codigo_tipo', 'descripcion_tipo']]
print('✓ pipeline_pandas definido (mismo alcance que el PoC)')

## 3. Tres ejecuciones de cada herramienta

Métricas que pide la guía, medidas de la carga a la exportación:
- **Tiempo total (s)** del pipeline.
- **Memoria pico (%)**: máximo de `psutil.virtual_memory().percent` durante la ejecución.
- **Throughput (registros/min)** = registros leídos / (tiempo total / 60).
- **Tasa de éxito de validaciones (%)** = controles aprobados / controles evaluados × 100.

In [ ]:
catalogo_sp = leer_catalogo_spark(spark)
EJECUCIONES = 3
metricas = []
for i in range(1, EJECUCIONES + 1):
    for herramienta in ('spark', 'pandas'):
        destino = f'/content/poc_{herramienta}_e{i}'
        gc.collect()
        with MonitorMemoria() as monitor:
            if herramienta == 'spark':
                p = PipelineEmergencias(spark, f'PoC_e{i}', destino, f'{RESULTADOS}/logs/poc_spark_e{i}.txt')
                p.run(RUTAS, catalogo_sp)
                m = p.metrics
            else:
                m = pipeline_pandas(RUTAS, destino, catalogo_pd)
        metricas.append({'herramienta': herramienta, 'ejecucion': i, 'tiempo_total_s': m['tiempo_total_s'],
                         'mem_pct_pico': monitor.pico, 'throughput_reg_min': m['leidos'] / (m['tiempo_total_s'] / 60),
                         'tasa_validaciones_pct': m['tasa_validaciones_pct'], 'leidos': m['leidos'],
                         'aceptados': m['aceptados'], 'errores': m['errores']})
        print(f"✓ Ejecución {i} · {herramienta}: {m['tiempo_total_s']:.2f} s, pico {monitor.pico:.1f} %")
metricas = pd.DataFrame(metricas)
metricas

### 3.1 Promedios y comparación

In [ ]:
COLS = ['tiempo_total_s', 'mem_pct_pico', 'throughput_reg_min', 'tasa_validaciones_pct']
display(metricas.groupby('herramienta')[COLS].agg(['mean', 'std']).round(2))
medias = metricas.groupby('herramienta')[COLS].mean()
ps, pp = medias.loc['spark'], medias.loc['pandas']
if ps['tiempo_total_s'] > pp['tiempo_total_s']:
    print(f"PySpark fue {ps['tiempo_total_s'] / pp['tiempo_total_s']:.1f} veces MÁS LENTO que Pandas con este volumen.")
else:
    print(f"PySpark fue {pp['tiempo_total_s'] / ps['tiempo_total_s']:.1f} veces más rápido que Pandas.")
print(f"Memoria pico: PySpark {ps['mem_pct_pico']:.1f} % · Pandas {pp['mem_pct_pico']:.1f} %")
print(f"Throughput: PySpark {ps['throughput_reg_min']:,.0f} · Pandas {pp['throughput_reg_min']:,.0f} registros/min")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
colores = ['#F58518', '#4C78A8']
for eje, col, titulo in ((ejes[0], 'tiempo_total_s', 'Tiempo total (s)'), (ejes[1], 'mem_pct_pico', 'Memoria pico del sistema (%)')):
    g = metricas.groupby('herramienta')[col].agg(['mean', 'std']).loc[['spark', 'pandas']]
    eje.bar(['PySpark', 'Pandas'], g['mean'], yerr=g['std'], capsize=5, color=colores)
    eje.set_title(titulo)
    eje.grid(True, axis='y', alpha=0.3)
fig.suptitle('PoC: PySpark frente a Pandas (media de 3 ejecuciones ± desv. estándar)', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Escalabilidad: 1 y 2 años de datos

Tres meses son ~27 000 registros; un año son ~4 veces y dos años ~8 veces más. Para medirlo sin inventar se crean
volúmenes **sintéticos** replicando los 3 archivos 4 y 8 veces (a cada copia se le cambia `NUMERO_INCIDENTE` para que
no sea un duplicado exacto). Se ejecuta una vez cada pipeline por volumen.

In [ ]:
def replicar(factor):
    carpeta = f'/content/escala_x{factor}'
    shutil.rmtree(carpeta, ignore_errors=True)
    os.makedirs(carpeta)
    rutas = []
    for a, r in zip(ARCHIVOS, RUTAS):
        base = pd.read_csv(r, sep=';', dtype=str, encoding='utf-8-sig', keep_default_na=False)
        copias = [base.assign(NUMERO_INCIDENTE=base['NUMERO_INCIDENTE'] + (f'-c{k}' if k else '')) for k in range(factor)]
        destino = f'{carpeta}/{a}'
        pd.concat(copias).to_csv(destino, sep=';', index=False, encoding='utf-8-sig')
        rutas.append(destino)
    return rutas


escala = []
for factor in (1, 4, 8):
    rutas = RUTAS if factor == 1 else replicar(factor)
    for herramienta in ('spark', 'pandas'):
        gc.collect()
        with MonitorMemoria() as monitor:
            if herramienta == 'spark':
                p = PipelineEmergencias(spark, f'Escala_x{factor}', f'/content/esc_spark_x{factor}',
                                        f'/content/esc_logs/esc_x{factor}.txt')
                p.run(rutas, catalogo_sp)
                m = p.metrics
            else:
                m = pipeline_pandas(rutas, f'/content/esc_pandas_x{factor}', catalogo_pd)
        escala.append({'factor': factor, 'meses_equivalentes': 3 * factor, 'herramienta': herramienta,
                       'registros': m['leidos'], 'tiempo_s': m['tiempo_total_s'], 'mem_pct_pico': monitor.pico,
                       'sintetico': factor > 1})
        print(f"✓ x{factor} · {herramienta}: {m['leidos']:,} registros en {m['tiempo_total_s']:.2f} s")
escala = pd.DataFrame(escala)
escala

### 4.1 Tendencia y proyección

Se ajusta una recta `tiempo = a + b × registros` con los tres puntos medidos. Doce y 24 meses están dentro del rango
medido (datos sintéticos); 48 meses es una **estimación teórica** fuera de ese rango.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
proyeccion = []
for herramienta, color in (('spark', '#F58518'), ('pandas', '#4C78A8')):
    d = escala[escala['herramienta'] == herramienta]
    b, a = np.polyfit(d['registros'], d['tiempo_s'], 1)
    ax.plot(d['registros'], d['tiempo_s'], 'o-', color=color, label=f'{herramienta.capitalize()} (medido)')
    x = np.linspace(d['registros'].min(), d['registros'].max() * 2, 50)
    ax.plot(x, a + b * x, '--', color=color, alpha=0.5, label=f'{herramienta.capitalize()} (tendencia)')
    base = int(d.loc[d['factor'] == 1, 'registros'].iloc[0])
    for meses in (12, 24, 48):
        registros = base * meses // 3
        proyeccion.append({'herramienta': herramienta, 'meses': meses, 'registros': registros,
                           'tiempo_estimado_s': round(a + b * registros, 2), 'segundos_por_10k_registros': round(b * 1e4, 3),
                           'tipo': 'dentro del rango medido (sintético)' if meses <= 24 else 'estimación teórica'})
ax.set_xlabel('Registros procesados')
ax.set_ylabel('Tiempo total (s)')
ax.set_title('Escalabilidad del pipeline: PySpark vs Pandas')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
proyeccion = pd.DataFrame(proyeccion)
proyeccion

## 5. Guardar las métricas en Drive

In [ ]:
metricas.to_csv(f'{MEDICIONES}/poc_metricas.csv', index=False)
escala.to_csv(f'{MEDICIONES}/escalabilidad.csv', index=False)
proyeccion.to_csv(f'{MEDICIONES}/proyeccion.csv', index=False)
print('✓ Guardado en', MEDICIONES, '→', sorted(os.listdir(MEDICIONES)))
spark.stop()

## 6. Interpretación

- Con 3 meses de datos, PySpark carga con costos fijos (JVM, planificación, coordinación) que Pandas no tiene; si
  resulta más lento o usa más memoria, así se reporta.
- La tasa de éxito de validaciones debe ser igual en ambas herramientas, porque aplican los mismos controles.
- La ventaja de PySpark aparece cuando los datos ya no caben en la memoria de una máquina: puede repartir el trabajo
  entre varios nodos sin cambiar el código. Colab tiene un solo nodo, así que eso no se mide aquí y queda como
  justificación técnica de la arquitectura.